# 🧠 تدريب الذكاء الاصطناعي لبروتوكول BARH - MEGA DATASET
## نموذج LSTM + Deep Q-Learning مع أكبر dataset للشبكات (97,190 عينة)

---

### 📊 MEGA DATASET المتوفر:
- **97,190 عينة** من بيانات حقيقية 100%
- **5 مصادر مختلفة**: ESP32 IoT + Internet Speed + Network Traffic + DoS + Anomaly
- **11 خصائص** أساسية موحدة
- **6 أنواع تصحيح** BARH
- **حجم الملف**: 11.5 MB

### 🎯 الهدف:
تدريب نماذج ذكاء اصطناعي متقدمة لبروتوكول BARH للحصول على:
- دقة LSTM > 85%
- معدل نجاح DQN > 70%
- تحسن أداء الشبكة 40-65%

In [ ]:
# 🔧 تثبيت المكتبات المطلوبة
!pip install tensorflow==2.13.0
!pip install torch torchvision torchaudio
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install wandb tensorboard

print("✅ تم تثبيت جميع المكتبات بنجاح!")

In [ ]:
# 📚 استيراد المكتبات
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque
import random
import warnings
import time
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# إعداد البذور للتكرار
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)
random.seed(42)

print("✅ تم تحميل جميع المكتبات بنجاح!")
print(f"🔥 TensorFlow GPU: {tf.config.list_physical_devices('GPU')}")
print(f"🔥 PyTorch GPU: {torch.cuda.is_available()}")

## 📊 تحميل ومعالجة MEGA DATASET

In [ ]:
# 📂 تحميل MEGA DATASET
# تأكد من رفع ملف MEGA_NETWORK_DATASET.csv إلى Colab أولاً

print("🔄 تحميل MEGA NETWORK DATASET...")
try:
    data = pd.read_csv('MEGA_NETWORK_DATASET.csv')
    print(f"✅ تم تحميل البيانات بنجاح!")
    print(f"📊 عدد العينات: {len(data):,}")
    print(f"📋 عدد الأعمدة: {len(data.columns)}")
    print(f"💾 حجم البيانات في الذاكرة: {data.memory_usage(deep=True).sum() / (1024**2):.1f} MB")
except FileNotFoundError:
    print("❌ لم يتم العثور على ملف MEGA_NETWORK_DATASET.csv")
    print("📋 تأكد من رفع الملف إلى Google Colab أولاً")
    raise

# عرض معلومات أساسية
print(f"\n📋 أول 5 صفوف من البيانات:")
display(data.head())

print(f"\n📊 معلومات البيانات:")
print(data.info())

In [ ]:
# 🔍 تحليل جودة البيانات
print("🔍 تحليل جودة MEGA DATASET...")

# فحص القيم المفقودة
missing_values = data.isnull().sum()
if missing_values.sum() > 0:
    print(f"⚠️ القيم المفقودة:")
    print(missing_values[missing_values > 0])
else:
    print("✅ لا توجد قيم مفقودة")

# فحص القيم المكررة
duplicates = data.duplicated().sum()
print(f"🔄 القيم المكررة: {duplicates:,}")

# تحديد الخصائص والنتائج
feature_cols = [
    'download_speed', 'upload_speed', 'latency', 'packet_loss', 'jitter',
    'bandwidth_utilization', 'connected_devices', 'concurrent_connections', 
    'time_of_day', 'network_type_encoded', 'location_encoded'
]

target_cols = [
    'buffer_optimization', 'connection_pooling', 'route_optimization',
    'adaptive_compression', 'parallel_connections', 'predictive_prefetch'
]

print(f"\n✅ الخصائص ({len(feature_cols)}): {feature_cols}")
print(f"✅ النتائج ({len(target_cols)}): {target_cols}")

# التحقق من وجود جميع الأعمدة
missing_features = [col for col in feature_cols if col not in data.columns]
missing_targets = [col for col in target_cols if col not in data.columns]

if missing_features:
    print(f"❌ خصائص مفقودة: {missing_features}")
if missing_targets:
    print(f"❌ نتائج مفقودة: {missing_targets}")

if not missing_features and not missing_targets:
    print("✅ جميع الأعمدة المطلوبة متوفرة!")

In [ ]:
# 📊 إحصائيات وتوزيع البيانات
print("📊 إحصائيات MEGA DATASET:")

# إحصائيات الخصائص
print("\n🔹 إحصائيات الخصائص:")
display(data[feature_cols].describe())

# توزيع التصحيحات
print("\n🔧 توزيع تصحيحات BARH:")
for col in target_cols:
    count = data[col].sum()
    percentage = count / len(data) * 100
    print(f"  {col}: {count:,} ({percentage:.1f}%)")

# توزيع جودة الشبكة
if 'network_quality' in data.columns:
    print("\n🌐 توزيع جودة الشبكة:")
    quality_dist = data['network_quality'].value_counts()
    for quality, count in quality_dist.items():
        percentage = count / len(data) * 100
        print(f"  {quality}: {count:,} ({percentage:.1f}%)")

# توزيع أنواع الشبكات
print("\n📡 توزيع أنواع الشبكات:")
network_dist = data['network_type_encoded'].value_counts().sort_index()
network_names = ['WiFi', '4G', '5G', 'LAN', 'Satellite']
for i, count in network_dist.items():
    name = network_names[int(i)] if int(i) < len(network_names) else f'Type_{int(i)}'
    percentage = count / len(data) * 100
    print(f"  {name}: {count:,} ({percentage:.1f}%)")

In [ ]:
# 📈 رسوم بيانية للبيانات
plt.style.use('default')
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('توزيع معاملات MEGA DATASET', fontsize=16, fontweight='bold')

# رسم أهم 6 معاملات
important_features = ['download_speed', 'upload_speed', 'latency', 'packet_loss', 'jitter', 'bandwidth_utilization']
colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']

for i, (feature, color) in enumerate(zip(important_features, colors)):
    row, col = i // 3, i % 3
    axes[row, col].hist(data[feature], bins=50, alpha=0.7, color=color, edgecolor='black')
    axes[row, col].set_title(f'{feature.replace("_", " ").title()}')
    axes[row, col].set_xlabel('القيمة')
    axes[row, col].set_ylabel('التكرار')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# رسم توزيع التصحيحات
plt.figure(figsize=(12, 6))
correction_counts = [data[col].sum() for col in target_cols]
correction_names = [col.replace('_', '\n') for col in target_cols]

bars = plt.bar(correction_names, correction_counts, color=['skyblue', 'lightgreen', 'lightcoral', 'gold', 'plum', 'lightsalmon'])
plt.title('توزيع تصحيحات BARH في MEGA DATASET', fontsize=14, fontweight='bold')
plt.xlabel('نوع التصحيح')
plt.ylabel('عدد العينات')
plt.xticks(rotation=45)

# إضافة قيم على الأعمدة
for bar, count in zip(bars, correction_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
             f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 تم عرض الرسوم البيانية للبيانات")

## 🧠 تدريب نموذج LSTM المتقدم

In [ ]:
# 🏗️ بناء نموذج LSTM متقدم لـ MEGA DATASET
class MegaBARHLSTM:
    """نموذج LSTM متقدم لـ MEGA DATASET"""
    
    def __init__(self, sequence_length=10, features=11):
        self.sequence_length = sequence_length
        self.features = features
        self.model = None
        self.scaler = StandardScaler()
        self.history = None
        
    def build_model(self):
        """بناء نموذج LSTM متقدم"""
        
        print(f"🏗️ بناء نموذج LSTM للـ MEGA DATASET...")
        
        model = Sequential([
            # طبقة LSTM الأولى - معالجة التسلسلات الطويلة
            LSTM(128, return_sequences=True, input_shape=(self.sequence_length, self.features),
                 dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            
            # طبقة LSTM الثانية - استخراج الأنماط المعقدة
            LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            
            # طبقة LSTM الثالثة - التركيز على المعلومات المهمة
            LSTM(32, return_sequences=False, dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            
            # طبقات Dense للتصنيف المتقدم
            Dense(64, activation='relu'),
            BatchNormalization(),
            Dropout(0.3),
            
            Dense(32, activation='relu'),
            BatchNormalization(),
            Dropout(0.2),
            
            Dense(16, activation='relu'),
            Dropout(0.1),
            
            # طبقة الإخراج - 6 أنواع تصحيح BARH
            Dense(6, activation='sigmoid', name='barh_corrections')
        ])
        
        # تجميع النموذج مع محسن متقدم
        model.compile(
            optimizer=Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
            loss='binary_crossentropy',
            metrics=['accuracy', 'precision', 'recall']
        )
        
        self.model = model
        
        print(f"✅ تم بناء النموذج بنجاح!")
        print(f"📊 معاملات النموذج: {model.count_params():,}")
        
        return model
    
    def prepare_sequences(self, data, feature_cols, target_cols):
        """تحضير البيانات للتسلسل الزمني"""
        
        print(f"🔄 تحضير التسلسلات الزمنية...")
        
        # استخراج الخصائص والنتائج
        features = self.scaler.fit_transform(data[feature_cols])
        targets = data[target_cols].values
        
        # إنشاء التسلسلات
        X, y = [], []
        
        print(f"📊 إنشاء تسلسلات بطول {self.sequence_length}...")
        
        for i in range(len(features) - self.sequence_length):
            X.append(features[i:(i + self.sequence_length)])
            y.append(targets[i + self.sequence_length])
            
            if (i + 1) % 10000 == 0:
                print(f"   تم إنشاء {i + 1:,} تسلسل...")
        
        X, y = np.array(X), np.array(y)
        
        print(f"✅ تم إنشاء {len(X):,} تسلسل")
        print(f"📊 شكل البيانات: X={X.shape}, y={y.shape}")
        
        return X, y

# إنشاء نموذج LSTM
lstm_model = MegaBARHLSTM(sequence_length=10, features=11)
model = lstm_model.build_model()

# عرض هيكل النموذج
print("\n🏗️ هيكل النموذج:")
model.summary()

In [ ]:
# 📊 تحضير البيانات للتدريب
print("🔄 تحضير MEGA DATASET للتدريب...")

# تحضير التسلسلات
X, y = lstm_model.prepare_sequences(data, feature_cols, target_cols)

# تقسيم البيانات
print(f"\n📊 تقسيم البيانات...")
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y.sum(axis=1) > 0)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.111, random_state=42, stratify=y_temp.sum(axis=1) > 0)

print(f"✅ تقسيم البيانات:")
print(f"   📈 التدريب: {X_train.shape[0]:,} عينة ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   📊 التحقق: {X_val.shape[0]:,} عينة ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"   🧪 الاختبار: {X_test.shape[0]:,} عينة ({X_test.shape[0]/len(X)*100:.1f}%)")

# إحصائيات التوزيع
print(f"\n📊 توزيع التصحيحات في بيانات التدريب:")
for i, col in enumerate(target_cols):
    train_count = y_train[:, i].sum()
    train_pct = train_count / len(y_train) * 100
    print(f"   {col}: {train_count:,} ({train_pct:.1f}%)")

In [ ]:
# 🚀 تدريب نموذج LSTM مع MEGA DATASET
print("🚀 بدء تدريب نموذج LSTM على MEGA DATASET...")

# إعداد callbacks متقدمة
callbacks = [
    # إيقاف مبكر لتجنب Overfitting
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1,
        min_delta=0.001
    ),
    
    # تقليل معدل التعلم عند التوقف
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-7,
        verbose=1,
        min_delta=0.001
    ),
    
    # حفظ أفضل نموذج
    ModelCheckpoint(
        'best_mega_barh_lstm.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
        save_weights_only=False
    )
]

# بدء التدريب
start_time = time.time()

history = model.fit(
    X_train, y_train,
    batch_size=64,  # حجم أكبر للـ MEGA DATASET
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

training_time = time.time() - start_time
lstm_model.history = history

print(f"\n✅ انتهى التدريب!")
print(f"⏰ وقت التدريب: {training_time/60:.1f} دقيقة")
print(f"📊 عدد العصور المكتملة: {len(history.history['loss'])}")